# <a id='toc1_'></a>[Imports](#toc0_)

In [ ]:
import torch
import evaluate
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.metrics import classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    AutoModelForSequenceClassification
)
from seqeval.metrics import classification_report

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("Current device:", torch.cuda.current_device())
else:
    print("Running on CPU")

CUDA available: True
GPU name: NVIDIA GeForce RTX 4080
Current device: 0


In [3]:
# Load the XML data and convert it to a DataFrame
def xml_to_rows(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    rows = []

    for sentence in root.findall("sentence"):
        sid = sentence.get("id")
        text = sentence.find("text").text

        aspect_terms = sentence.find("aspectTerms")

        if aspect_terms is not None:
            for aspect in aspect_terms.findall("aspectTerm"):
                rows.append({
                    "id": sid,
                    "sentence": text,
                    "aspect": aspect.get("term"),
                    "polarity": aspect.get("polarity")
                })

    return pd.DataFrame(rows)

In [4]:
myXML = xml_to_rows("./Restaurants_Train_v2.xml")

In [5]:
myXML.shape

(3693, 4)

In [6]:
myXML.head()

,id,sentence,aspect,polarity
0,3121,But the staff was so horrible to us.,staff,negative
1,2777,"To be completely fair, the only redeeming fact...",food,positive
2,1634,"The food is uniformly exceptional, with a very...",food,positive
3,1634,"The food is uniformly exceptional, with a very...",kitchen,positive
4,1634,"The food is uniformly exceptional, with a very...",menu,neutral


In [7]:
csv_df = pd.read_csv("./Laptop_Train_v2.csv", encoding="utf-8")

csv_df = csv_df[[
    "id",
    "Sentence",
    "Aspect Term",
    "polarity"
]]

csv_df.columns = ["id", "sentence", "aspect", "polarity"]

In [8]:
csv_df.head()

,id,sentence,aspect,polarity
0,2339,I charge it at night and skip taking the cord ...,cord,neutral
1,2339,I charge it at night and skip taking the cord ...,battery life,positive
2,1316,The tech guy then said the service center does...,service center,negative
3,1316,The tech guy then said the service center does...,"""sales"" team",negative
4,1316,The tech guy then said the service center does...,tech guy,neutral


In [9]:
csv_df.shape

(2358, 4)

# <a id='toc2_'></a>[Merge the datasets](#toc0_)

In [10]:
final_df = pd.concat([csv_df, myXML], ignore_index=True)

In [11]:
final_df.head()

,id,sentence,aspect,polarity
0,2339,I charge it at night and skip taking the cord ...,cord,neutral
1,2339,I charge it at night and skip taking the cord ...,battery life,positive
2,1316,The tech guy then said the service center does...,service center,negative
3,1316,The tech guy then said the service center does...,"""sales"" team",negative
4,1316,The tech guy then said the service center does...,tech guy,neutral


In [12]:
final_df["sentence"].iloc[1]

'I charge it at night and skip taking the cord with me because of the good battery life.'

In [13]:
final_df["sentence"].iloc[1]

'I charge it at night and skip taking the cord with me because of the good battery life.'

In [14]:
final_df.shape

(6051, 4)

In [15]:
print(final_df["polarity"].unique())

<ArrowStringArray>
['neutral', 'positive', 'negative', 'conflict']
Length: 4, dtype: str


# <a id='toc3_'></a>[Preprocess](#toc0_)

In [16]:
final_df["polarity"] = final_df["polarity"].str.strip()
final_df["aspect"] = final_df["aspect"].str.replace('"', '')

# <a id='toc4_'></a>[Take 1000 of dataset](#toc0_)

In [17]:
final_df = final_df.head(6000)

In [18]:
final_df.shape

(6000, 4)

# <a id='toc5_'></a>[BIO tagging](#toc0_)

In [ ]:
import string

# ---------------------------
# TOKENIZER (clean + simple)
# ---------------------------
def tokenize(text):
    return [
        t.strip(string.punctuation).lower()
        for t in text.split()
        if t.strip(string.punctuation)
    ]

# ---------------------------
# FIND ASPECT SPAN
# ---------------------------
def find_span(tokens, aspect_tokens):
    n, m = len(tokens), len(aspect_tokens)

    for i in range(n - m + 1):
        if tokens[i:i+m] == aspect_tokens:
            return i, i + m - 1

    return -1, -1

# ---------------------------
# BIOES TAGGING (PAPER STYLE)
# ---------------------------
def build_tags(tokens, start, end, polarity):
    tags = ["O"] * len(tokens)

    length = end - start + 1

    # SINGLE WORD → S
    if length == 1:
        tags[start] = f"S-{polarity}"

    # MULTI WORD → B I E
    else:
        tags[start] = f"B-{polarity}"

        for i in range(start + 1, end):
            tags[i] = f"I-{polarity}"

        tags[end] = f"E-{polarity}"

    return tags

# ---------------------------
# MAIN CONVERTER
# ---------------------------
def convert(df):
    dataset = []

    for _, row in df.iterrows():
        sentence = row["sentence"]
        aspect = row["aspect"]
        polarity = row["polarity"].upper()  # POSITIVE/NEGATIVE/NEUTRAL/CONFLICT

        tokens = tokenize(sentence)
        aspect_tokens = tokenize(aspect)

        start, end = find_span(tokens, aspect_tokens)

        tags = ["O"] * len(tokens)

        if start != -1:
            tags = build_tags(tokens, start, end, polarity)

        dataset.append(list(zip(tokens, tags)))

    return dataset

# Model training 

In [ ]:
# =========================================================
# 1. LABELS (BIOES + SENTIMENT)
# =========================================================
labels = [
    "O",
    "B-POSITIVE", "I-POSITIVE", "E-POSITIVE", "S-POSITIVE",
    "B-NEGATIVE", "I-NEGATIVE", "E-NEGATIVE", "S-NEGATIVE",
    "B-NEUTRAL", "I-NEUTRAL", "E-NEUTRAL", "S-NEUTRAL",
    "B-CONFLICT", "I-CONFLICT", "E-CONFLICT", "S-CONFLICT"
]

label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

# =========================================================
# 2. TOKENIZER
# =========================================================
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# =========================================================
# 3. CONVERT YOUR BIOES DATA -> HF DATASET
# =========================================================
def prepare_dataset(bioes_data):
    tokens_list = []
    labels_list = []

    for sent in bioes_data:
        tokens_list.append([t for t, l in sent])
        labels_list.append([l for t, l in sent])

    return Dataset.from_dict({
        "tokens": tokens_list,
        "labels": labels_list
    })

# =========================================================
# 4. ALIGN LABELS TO BERT TOKENS
# =========================================================
def tokenize_and_align(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True
    )

    labels_batch = []

    for i, word_labels in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label2id[word_labels[word_idx]])

            else:
                # subword handling
                label = word_labels[word_idx]
                if label.startswith("I"):
                    label_ids.append(label2id[label])
                else:
                    label_ids.append(-100)

            previous_word_idx = word_idx

        labels_batch.append(label_ids)

    tokenized_inputs["labels"] = labels_batch
    return tokenized_inputs

# =========================================================
# 5. LOAD DATA
# =========================================================
bioes_data = convert(final_df)   
dataset = prepare_dataset(bioes_data)

dataset = dataset.train_test_split(test_size=0.2)

tokenized_dataset = dataset.map(tokenize_and_align, batched=True)

# =========================================================
# 6. MODEL
# =========================================================
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

# =========================================================
# 7. TRAINING SETUP
# =========================================================
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./new_absa_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    report_to="none"
)

# =========================================================
# 8. METRICS
# =========================================================
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_preds = []

    for pred, lab in zip(predictions, labels):
        temp_labels = []
        temp_preds = []

        for p_i, l_i in zip(pred, lab):
            if l_i != -100:
                temp_labels.append(id2label[int(l_i)])
                temp_preds.append(id2label[int(p_i)])

        true_labels.append(temp_labels)
        true_preds.append(temp_preds)

    return {
        "report": classification_report(true_labels, true_preds, digits=4)
    }

# =========================================================
# 9. TRAINER
# =========================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# =========================================================
# 10. TRAIN
# =========================================================
trainer.train()

Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

Epoch,Training Loss,Validation Loss,Report
1,No log,0.241070,precision recall f1-score support CONFLICT 0.0000 0.0000 0.0000 33 NEGATIVE 0.4780 0.2970 0.3664 330 NEUTRAL 0.0000 0.0000 0.0000 208 POSITIVE 0.4319 0.3258 0.3715 623 micro avg 0.4453 0.2521 0.3219 1194 macro avg 0.2275 0.1557 0.1845 1194 weighted avg 0.3575 0.2521 0.2951 1194
2,0.313016,0.226043,precision recall f1-score support CONFLICT 0.0000 0.0000 0.0000 33 NEGATIVE 0.4009 0.5152 0.4509 330 NEUTRAL 0.2683 0.0529 0.0884 208 POSITIVE 0.4194 0.4510 0.4346 623 micro avg 0.4070 0.3869 0.3967 1194 macro avg 0.2722 0.2548 0.2435 1194 weighted avg 0.3764 0.3869 0.3668 1194
3,0.313016,0.222873,precision recall f1-score support CONFLICT 0.0000 0.0000 0.0000 33 NEGATIVE 0.5000 0.3879 0.4369 330 NEUTRAL 0.2576 0.0817 0.1241 208 POSITIVE 0.4833 0.3242 0.3881 623 micro avg 0.4689 0.2906 0.3588 1194 macro avg 0.3102 0.1985 0.2373 1194 weighted avg 0.4352 0.2906 0.3449 1194
4,0.196353,0.230859,precision recall f1-score support CONFLICT 0.0000 0.0000 0.0000 33 NEGATIVE 0.4630 0.3606 0.4055 330 NEUTRAL 0.2243 0.1154 0.1524 208 POSITIVE 0.4585 0.3018 0.3640 623 micro avg 0.4276 0.2772 0.3364 1194 macro avg 0.2865 0.1944 0.2305 1194 weighted avg 0.4063 0.2772 0.3285 1194
5,0.164857,0.238434,precision recall f1-score support CONFLICT 0.0000 0.0000 0.0000 33 NEGATIVE 0.4618 0.3848 0.4198 330 NEUTRAL 0.2421 0.1106 0.1518 208 POSITIVE 0.4185 0.3628 0.3887 623 micro avg 0.4132 0.3149 0.3574 1194 macro avg 0.2806 0.2145 0.2401 1194 weighted avg 0.3882 0.3149 0.3453 1194


...

c:\Users\lalis\miniconda3\envs\ai_conda\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\lalis\miniconda3\envs\ai_conda\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1500, training_loss=0.22474213155110678, metrics={'train_runtime': 69.4862, 'train_samples_per_second': 345.392, 'train_steps_per_second': 21.587, 'total_flos': 575412654192672.0, 'train_loss': 0.22474213155110678, 'epoch': 5.0})

In [23]:
tokenizer.save_pretrained("./new_absa_model")
model.save_pretrained("./new_absa_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Inference

In [26]:
from transformers import AutoTokenizer

# =========================================================
# 1. LOAD MODEL + TOKENIZER (NOW CORRECT)
# =========================================================
model_path = "./new_absa_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

model.eval()

# =========================================================
# 2. LABEL MAP (must match training)
# =========================================================
labels = [
    "O",
    "B-POSITIVE", "I-POSITIVE", "E-POSITIVE", "S-POSITIVE",
    "B-NEGATIVE", "I-NEGATIVE", "E-NEGATIVE", "S-NEGATIVE",
    "B-NEUTRAL", "I-NEUTRAL", "E-NEUTRAL", "S-NEUTRAL"
]

id2label = {i: l for i, l in enumerate(labels)}

# =========================================================
# 3. PREDICT TOKEN LABELS
# =========================================================
def predict(sentence):
    tokens = sentence.lower().split()

    inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    preds = torch.argmax(outputs.logits, dim=2)[0].cpu().numpy()
    word_ids = inputs.word_ids()

    results = []
    prev_word = None

    for i, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue

        if word_idx != prev_word:
            token = tokens[word_idx]
            label = id2label[int(preds[i])]
            results.append((token, label))

        prev_word = word_idx

    return results

# =========================================================
# 4. EXTRACT ASPECTS (BIOES decoding)
# =========================================================
def extract_aspects(predictions):
    aspects = []
    current = []

    for token, label in predictions:

        if label.startswith("B-") or label.startswith("S-"):
            if current:
                aspects.append(current)
            current = [(token, label)]

        elif label.startswith("I-") or label.startswith("E-"):
            current.append((token, label))

        else:
            if current:
                aspects.append(current)
                current = []

    if current:
        aspects.append(current)

    return aspects

# =========================================================
# 5. FORMAT FINAL OUTPUT
# =========================================================
def format_output(aspects):
    results = []

    for asp in aspects:
        words = [w for w, l in asp]
        label = asp[0][1]

        if "POSITIVE" in label:
            sentiment = "POSITIVE"
        elif "NEGATIVE" in label:
            sentiment = "NEGATIVE"
        else:
            sentiment = "NEUTRAL"

        results.append({
            "aspect": " ".join(words),
            "sentiment": sentiment
        })

    return results

# =========================================================
# 6. FULL PIPELINE
# =========================================================
def analyze(sentence):
    preds = predict(sentence)
    aspects = extract_aspects(preds)
    return format_output(aspects)

# =========================================================
# 7. TEST
# =========================================================
if __name__ == "__main__":
    sentence = "The phone has a great camera, but the screen brightness is a bit disappointing."

    print(analyze(sentence))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'aspect': 'screen brightness', 'sentiment': 'NEGATIVE'}]


# Convert file to .py

In [25]:
from nbconvert import ScriptExporter
import nbformat

with open("absa-e2e.ipynb") as f:
    nb = nbformat.read(f, as_version=4)

exporter = ScriptExporter()
script, _ = exporter.from_notebook_node(nb)

with open("mypythonfile.py", "w") as f:
    f.write(script)